# Example 03: Policies in Practice

**Goal:** Configure and test Omnigent's policy system — spend caps, tool limits, approval gates.

**Key concept:** Policies are declarative guardrails that return ALLOW / DENY / ASK. They evaluate in order and a DENY from any policy blocks the action.

## 1. Policy Declaration Syntax

In [ ]:
policy_template = '''
# Every policy uses the same structure:
policy_name:
  type: function                           # always "function"
  handler: omnigent.policies.builtins.XXX   # dotted Python import path
  factory_params:                           # optional constructor args
    key: value
'''
print("Policy template:")
print(policy_template)

## 2. Policy 1: Cost Budget

Caps total spending for a session. ASKs at warning thresholds, DENYs at cap.

In [ ]:
budget_policy = '''
policies:
  budget:
    type: function
    handler: omnigent.policies.builtins.cost.cost_budget
    factory_params:
      max_cost_usd: 5.00
      ask_thresholds_usd: [3.00]
'''
print("Budget policy:")
print(budget_policy)
print("Effect: Silent until $3.00 → ASK at $3.00 → DENY at $5.00")

## 3. Policy 2: Tool Call Rate Limit

In [ ]:
rate_limit = '''
policies:
  rate_limit:
    type: function
    handler: omnigent.policies.builtins.safety.max_tool_calls_per_session
    factory_params:
      limit: 50
'''
print("Rate limit policy:")
print(rate_limit)
print("Effect: After 50 tool calls, every subsequent call is DENIED")

## 4. Policy 3: Shell Operation Approval

In [ ]:
shell_approval = '''
policies:
  approve_ops:
    type: function
    handler: omnigent.policies.builtins.safety.ask_on_os_tools
'''
print("Shell approval policy:")
print(shell_approval)
print("Effect: EVERY sys_os_read/write/edit/shell call pauses for user approval")

## 5. Composing Policies: A Complete Agent

In [ ]:
full_agent = '''
name: governed_coder
description: A coding agent with budget control and safety gates.

prompt: |
  You are a coding assistant. Write clean, tested code.
  Always explain your reasoning before making changes.

executor:
  harness: claude-sdk
  model: claude-sonnet-4-6

os_env:
  type: caller_process
  cwd: .
  sandbox:
    type: none

policies:
  # 1. Hard spend cap
  budget:
    type: function
    handler: omnigent.policies.builtins.cost.cost_budget
    factory_params:
      max_cost_usd: 2.00
      ask_thresholds_usd: [1.00]

  # 2. Tool call limit
  rate_limit:
    type: function
    handler: omnigent.policies.builtins.safety.max_tool_calls_per_session
    factory_params:
      limit: 100

  # 3. Block dangerous skills
  no_deploy:
    type: function
    handler: omnigent.policies.builtins.safety.block_skills
    factory_params:
      blocked: [deploy, destroy, provision]

# Guardrails: runner-level blast radius protection
guardrails:
  ask_timeout: 3600
  policies:
    blast_radius:
      type: function
      function:
        path: omnigent.inner.nessie.policies.blast_radius
        arguments:
          gate_pushes: true
'''

with open('agent_governed.yaml', 'w') as f:
    f.write(full_agent)
print("Created agent_governed.yaml")
print("This agent has: budget cap, rate limit, blocked skills, blast radius protection")

## 6. Server-Wide Policies (Admin)

Policies at the server level set the *floor* — agents can be stricter but not looser.

In [ ]:
server_policies = '''
# server_config.yaml (set by admin)
policies:
  team_sandbox:
    type: function
    handler: omnigent.policies.builtins.safety.enforce_sandbox
    factory_params:
      sandbox_type: linux_bwrap
      write_paths: [./output]
      allow_network: true

  team_budget:
    type: function
    handler: omnigent.policies.builtins.cost.cost_budget
    factory_params:
      max_cost_usd: 50.00
      ask_thresholds_usd: [25.00]
'''
print("Server-level policies (admin only):")
print(server_policies)

## 7. Policy Evaluation Order

```
SESSION policies (user-set, evaluated FIRST)
    ↓
AGENT policies (in agent.yaml, evaluated SECOND)
    ↓
SERVER policies (admin-set, evaluated LAST)
    ↓
DENY at any level → short-circuits everything below
```

**Key:** User can always add STRICTER limits (session level evaluates first).

**Next:** [Example 04 — Multi-Agent Orchestration](./example_04_multi_agent.ipynb)